# 04 — REVE (foundation model EEG)
`brain-bzh/reve-large` — transformer pre-addestrato su >60k ore di EEG, con positional encoding 4D adattivo a montaggi variabili. **Input a 200 Hz** (i dati vengono resamplati 256→200) + posizioni elettrodi da `brain-bzh/reve-positions`.

Qui REVE è usato come **feature extractor congelato + testa lineare** (fine-tuning leggero), subject-dependent.

**Requisiti**: `pip install transformers` e connessione internet al primo run per scaricare i pesi.

> ⚠️ L'API esatta di REVE (forma dell'output) va verificata al primo run: il wrapper `REVEClassifier` gestisce in modo difensivo output tensore/dict e costruisce la testa in modo lazy. Se il forward fallisce, controlla la firma di `model(eeg, positions)` sulla model card HF.

In [ ]:
# --- setup: rende importabili i moduli track3_*.py ---
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, matplotlib.pyplot as plt
import track3_config as C, track3_io as io, track3_preproc as P
print(C.summary())
assert C.DATA_ROOT is not None, C._no_data_msg()
import track3_train as T
import track3_models as M
device=C.get_device(); print('device:', device)

## 1. Test rapido di caricamento (1 soggetto, poche epoche)
Scarica i pesi la prima volta. Se qui va, il run completo è solo questione di tempo.

In [ ]:
df_reve, res_reve = T.run_subject_dependent(
    'reve', subjects=[1], model_kwargs=dict(freeze_backbone=True),
    train_kwargs=dict(epochs=30, patience=10, lr=1e-3, batch_size=16))
df_reve

## 2. Run completo su tutti i soggetti
Scommenta quando il test sopra è andato a buon fine.

In [ ]:
# df_reve, res_reve = T.run_subject_dependent(
#     'reve', model_kwargs=dict(freeze_backbone=True),
#     train_kwargs=dict(epochs=50, patience=15, lr=1e-3, batch_size=16))
# T.save_metrics(df_reve, 'reve')
# T.plot_per_subject(df_reve, 'reve'); plt.show()
# T.plot_confusion(res_reve, model_name='reve'); plt.show()